# 05 - SAS-Python Cross-Platform Validation

## Purpose

This notebook is a lightweight validation layer between independently generated Python and SAS outputs for the non-ICU RAAS project.

It compares exported CSV artifacts. It does not perform new statistical modeling, rebuild upstream summaries, or reinterpret the clinical analysis.


## Scope

The notebook is limited to four tasks:

1. Load Python output CSV files.
2. Load SAS output CSV files.
3. Apply minimal schema alignment where file naming conventions differ.
4. Compare values and summarize discrepancies.

The statistical analysis logic belongs upstream in notebooks `01`-`04b` and SAS programs `01`-`04`. This notebook should remain an output comparison layer only.


## Comparison Philosophy

Validation is organized by analysis stage:

| Stage | Python source | Python output | SAS source | SAS output |
|---|---|---|---|---|
| 04a unadjusted outcomes | `notebooks/04a_unadjusted_outcomes.ipynb` | `python/outputs/python_unadjusted_outcomes.csv` | `sas/programs/03_unadjusted_analysis.sas` | `sas/outputs/sas_unadjusted_outcomes.csv` |
| 04b multivariable model | `notebooks/04b_multivariable_outcomes.ipynb` | `python/outputs/python_logistic_parameters.csv` | `sas/programs/04_multivariable_logistic.sas` | `sas/outputs/sas_logistic_parameters.csv` |

Minimal normalization is acceptable when it makes independently generated outputs comparable, such as column renaming, type coercion, sorting before merge, and explicit term-name aliasing. Hidden fallback calculations and reconstructed model outputs are intentionally avoided.


## Known SAS-Python Implementation Differences

Flagged differences should be reviewed in context. Differences can arise from categorical reference levels, dummy-variable coding, numeric precision, rounding, convergence behavior, or quasi-complete separation in sparse categories.

The multivariable model includes `admission_type` categories with extreme odds ratios. These terms are especially sensitive to sparse cells and rare outcomes, so implementation-specific differences may appear even when the model specification is unchanged.


In [71]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


In [72]:
NOTEBOOKS_DIR = Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"
PROJECT_ROOT = NOTEBOOKS_DIR.parent

PYTHON_UNADJUSTED_PATH = PROJECT_ROOT / "python" / "outputs" / "python_unadjusted_outcomes.csv"
SAS_UNADJUSTED_PATH = PROJECT_ROOT / "sas" / "outputs" / "sas_unadjusted_outcomes.csv"
PYTHON_LOGISTIC_PATH = PROJECT_ROOT / "python" / "outputs" / "python_logistic_parameters.csv"
SAS_LOGISTIC_PATH = PROJECT_ROOT / "sas" / "outputs" / "sas_logistic_parameters.csv"

expected_files = pd.DataFrame(
    [
        ("04a", "Python", PYTHON_UNADJUSTED_PATH),
        ("04a", "SAS", SAS_UNADJUSTED_PATH),
        ("04b", "Python", PYTHON_LOGISTIC_PATH),
        ("04b", "SAS", SAS_LOGISTIC_PATH),
    ],
    columns=["stage", "platform", "path"],
)
expected_files["relative_path"] = expected_files["path"].map(lambda p: str(p.relative_to(PROJECT_ROOT)))
expected_files["available"] = expected_files["path"].map(Path.exists)
expected_files[["stage", "platform", "relative_path", "available"]]


,stage,platform,relative_path,available
0,04a,Python,python/outputs/python_unadjusted_outcomes.csv,True
1,04a,SAS,sas/outputs/sas_unadjusted_outcomes.csv,True
2,04b,Python,python/outputs/python_logistic_parameters.csv,True
3,04b,SAS,sas/outputs/sas_logistic_parameters.csv,True


In [73]:
def load_csv(path: Path, label: str, required: bool = False) -> pd.DataFrame | None:
    if path.exists():
        return pd.read_csv(path)

    message = f"TODO: `{path.relative_to(PROJECT_ROOT)}` is missing. Generate the {label} output before completing this validation section."
    if required:
        raise FileNotFoundError(message)

    display(Markdown(message))
    return None

python_unadjusted_raw = load_csv(PYTHON_UNADJUSTED_PATH, "Python 04a", required=True)
sas_unadjusted_raw = load_csv(SAS_UNADJUSTED_PATH, "SAS 03")
python_logistic_raw = load_csv(PYTHON_LOGISTIC_PATH, "Python 04b", required=True)
sas_logistic_raw = load_csv(SAS_LOGISTIC_PATH, "SAS 04")


## Validation 1: 04a Unadjusted Outcomes

**Python source**: `notebooks/04a_unadjusted_outcomes.ipynb`  
**SAS source**: `sas/programs/03_unadjusted_analysis.sas`  
**Files compared**: `python/outputs/python_unadjusted_outcomes.csv` vs `sas/outputs/sas_unadjusted_outcomes.csv`

Variables compared:

- `exposure`
- `exposure_group`
- `n`
- `deaths`
- `non_deaths`
- `mortality_proportion`
- `crude_odds`
- `crude_odds_ratio_vs_no_early_raas`

Tolerance: numeric absolute difference <= `1e-8`. String fields must match exactly after string coercion.


In [74]:
UNADJUSTED_COLUMN_RENAMES = {
    "crude_or_vs_no_raas": "crude_odds_ratio_vs_no_early_raas",
}

UNADJUSTED_KEY = ["exposure"]
UNADJUSTED_COMPARE_COLUMNS = [
    "exposure_group",
    "n",
    "deaths",
    "non_deaths",
    "mortality_proportion",
    "crude_odds",
    "crude_odds_ratio_vs_no_early_raas",
]
UNADJUSTED_NUMERIC_COLUMNS = [
    "n",
    "deaths",
    "non_deaths",
    "mortality_proportion",
    "crude_odds",
    "crude_odds_ratio_vs_no_early_raas",
]
UNADJUSTED_ABS_TOL = 1e-8


def prepare_unadjusted(df: pd.DataFrame, platform: str) -> pd.DataFrame:
    out = df.rename(columns=UNADJUSTED_COLUMN_RENAMES).copy()
    expected = UNADJUSTED_KEY + UNADJUSTED_COMPARE_COLUMNS
    missing = [col for col in expected if col not in out.columns]
    if missing:
        raise ValueError(f"{platform} unadjusted output is missing columns: {missing}")

    out = out[expected].copy()
    out["exposure"] = pd.to_numeric(out["exposure"], errors="coerce")
    for col in UNADJUSTED_NUMERIC_COLUMNS:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    return out.sort_values(UNADJUSTED_KEY).reset_index(drop=True)

python_unadjusted = prepare_unadjusted(python_unadjusted_raw, "Python")
sas_unadjusted = None if sas_unadjusted_raw is None else prepare_unadjusted(sas_unadjusted_raw, "SAS")

python_unadjusted


,exposure,exposure_group,n,deaths,non_deaths,mortality_proportion,crude_odds,crude_odds_ratio_vs_no_early_raas
0,0,No early RAAS inhibitor use,403961,2177,401784,0.005389,0.005418,1.000000
1,1,Early RAAS inhibitor use,56825,149,56676,0.002622,0.002629,0.485201


In [75]:
def compare_table(
    python_df: pd.DataFrame,
    sas_df: pd.DataFrame,
    key_columns: list[str],
    compare_columns: list[str],
    numeric_columns: list[str],
    abs_tolerance: float,
) -> pd.DataFrame:
    merged = python_df.merge(
        sas_df,
        on=key_columns,
        how="outer",
        suffixes=("_python", "_sas"),
        indicator=True,
    )

    rows = []
    for _, row in merged.iterrows():
        key_label = ", ".join(f"{key}={row[key]}" for key in key_columns)
        for variable in compare_columns:
            python_value = row.get(f"{variable}_python")
            sas_value = row.get(f"{variable}_sas")
            is_numeric = variable in numeric_columns

            if row["_merge"] != "both":
                absolute_difference = np.nan
                relative_difference = np.nan
                passed = False
            elif is_numeric:
                absolute_difference = abs(sas_value - python_value)
                denominator = max(abs(python_value), 1e-12)
                relative_difference = absolute_difference / denominator
                passed = bool(absolute_difference <= abs_tolerance)
            else:
                absolute_difference = np.nan
                relative_difference = np.nan
                passed = bool(str(python_value) == str(sas_value))

            rows.append(
                {
                    "key": key_label,
                    "variable": variable,
                    "python_value": python_value,
                    "sas_value": sas_value,
                    "absolute_difference": absolute_difference,
                    "relative_difference": relative_difference,
                    "pass": passed,
                }
            )
    return pd.DataFrame(rows)

if sas_unadjusted is None:
    display(Markdown("TODO: SAS unadjusted output is unavailable; comparison table was not generated."))
    unadjusted_comparison = pd.DataFrame()
else:
    unadjusted_comparison = compare_table(
        python_df=python_unadjusted,
        sas_df=sas_unadjusted,
        key_columns=UNADJUSTED_KEY,
        compare_columns=UNADJUSTED_COMPARE_COLUMNS,
        numeric_columns=UNADJUSTED_NUMERIC_COLUMNS,
        abs_tolerance=UNADJUSTED_ABS_TOL,
    )

unadjusted_comparison


,key,variable,python_value,sas_value,absolute_difference,relative_difference,pass
0,exposure=0,exposure_group,No early RAAS inhibitor use,No early RAAS inhibitor use,NaN,NaN,True
1,exposure=0,n,403961,403961,0.000000e+00,0.000000e+00,True
2,exposure=0,deaths,2177,2177,0.000000e+00,0.000000e+00,True
3,exposure=0,non_deaths,401784,401784,0.000000e+00,0.000000e+00,True
4,exposure=0,mortality_proportion,0.005389,0.005389,1.099815e-15,2.040800e-13,True
5,exposure=0,crude_odds,0.005418,0.005418,2.899590e-15,5.351442e-13,True
6,exposure=0,crude_odds_ratio_vs_no_early_raas,1.0,1.0,5.200285e-13,5.200285e-13,True
7,exposure=1,exposure_group,Early RAAS inhibitor use,Early RAAS inhibitor use,NaN,NaN,True
8,exposure=1,n,56825,56825,0.000000e+00,0.000000e+00,True
9,exposure=1,deaths,149,149,0.000000e+00,0.000000e+00,True


In [76]:
unadjusted_discrepancies = unadjusted_comparison.loc[
    unadjusted_comparison["pass"].eq(False)
].copy() if not unadjusted_comparison.empty else pd.DataFrame()

unadjusted_discrepancies


,key,variable,python_value,sas_value,absolute_difference,relative_difference,pass


## Validation 2: 04b Multivariable Model

**Python source**: `notebooks/04b_multivariable_outcomes.ipynb`  
**SAS source**: `sas/programs/04_multivariable_logistic.sas`  
**Files compared**: `python/outputs/python_logistic_parameters.csv` vs `sas/outputs/sas_logistic_parameters.csv`

Variables compared when exported by both platforms:

- `coefficient`
- `odds_ratio`
- `ci_lower`
- `ci_upper`
- `p_value`

`std_error` is not reconstructed in this notebook if a platform does not export it. This preserves the notebook's role as an artifact comparison layer rather than a model-output derivation layer.

Tolerances:

- Coefficients and p-values: absolute difference <= `1e-4`
- Odds ratios and confidence limits: absolute difference <= `1e-4`


In [77]:
LOGISTIC_COLUMN_RENAMES = {
    "odds_ratio_ci_lower": "ci_lower",
    "odds_ratio_ci_upper": "ci_upper",
}

TERM_ALIASES = {
    "anchor_year_group_2011 - 2013": "anchor_year_group_2011_2013",
    "anchor_year_group_2014 - 2016": "anchor_year_group_2014_2016",
    "anchor_year_group_2017 - 2019": "anchor_year_group_2017_2019",
    "anchor_year_group_2020 - 2022": "anchor_year_group_2020_2022",
}

LOGISTIC_KEY = ["term"]
LOGISTIC_REQUESTED_COLUMNS = [
    "coefficient",
    "std_error",
    "odds_ratio",
    "ci_lower",
    "ci_upper",
    "p_value",
]
LOGISTIC_NUMERIC_COLUMNS = LOGISTIC_REQUESTED_COLUMNS.copy()
LOGISTIC_ABS_TOLERANCES = {
    "coefficient": 1e-4,
    "std_error": 1e-4,
    "odds_ratio": 1e-4,
    "ci_lower": 1e-4,
    "ci_upper": 1e-4,
    "p_value": 1e-4,
}


def prepare_logistic(df: pd.DataFrame, platform: str) -> pd.DataFrame:
    out = df.rename(columns=LOGISTIC_COLUMN_RENAMES).copy()
    out = out.loc[:, ~out.columns.duplicated()].copy()
    if "term" not in out.columns:
        raise ValueError(f"{platform} logistic output is missing column: term")

    available_columns = [col for col in LOGISTIC_REQUESTED_COLUMNS if col in out.columns]
    out = out[["term"] + available_columns].copy()
    out["term"] = out["term"].replace(TERM_ALIASES).astype(str)
    for col in available_columns:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    return out.sort_values(LOGISTIC_KEY).reset_index(drop=True)

python_logistic = prepare_logistic(python_logistic_raw, "Python")
sas_logistic = None if sas_logistic_raw is None else prepare_logistic(sas_logistic_raw, "SAS")

logistic_compare_columns = [
    col for col in LOGISTIC_REQUESTED_COLUMNS
    if col in python_logistic.columns and sas_logistic is not None and col in sas_logistic.columns
]

missing_from_python = [col for col in LOGISTIC_REQUESTED_COLUMNS if col not in python_logistic.columns]
missing_from_sas = [] if sas_logistic is None else [col for col in LOGISTIC_REQUESTED_COLUMNS if col not in sas_logistic.columns]

pd.DataFrame(
    {
        "metric": LOGISTIC_REQUESTED_COLUMNS,
        "compared": [col in logistic_compare_columns for col in LOGISTIC_REQUESTED_COLUMNS],
        "available_in_python": [col in python_logistic.columns for col in LOGISTIC_REQUESTED_COLUMNS],
        "available_in_sas": [False if sas_logistic is None else col in sas_logistic.columns for col in LOGISTIC_REQUESTED_COLUMNS],
    }
)


,metric,compared,available_in_python,available_in_sas
0,coefficient,True,True,True
1,std_error,False,False,True
2,odds_ratio,True,True,True
3,ci_lower,True,True,True
4,ci_upper,True,True,True
5,p_value,True,True,True


In [78]:
if sas_logistic is None:
    display(Markdown("TODO: SAS logistic output is unavailable; comparison table was not generated."))
    logistic_comparison = pd.DataFrame()
else:
    logistic_rows = []
    for variable in logistic_compare_columns:
        metric_comparison = compare_table(
            python_df=python_logistic[["term", variable]],
            sas_df=sas_logistic[["term", variable]],
            key_columns=LOGISTIC_KEY,
            compare_columns=[variable],
            numeric_columns=[variable],
            abs_tolerance=LOGISTIC_ABS_TOLERANCES[variable],
        )
        logistic_rows.append(metric_comparison)
    logistic_comparison = pd.concat(logistic_rows, ignore_index=True) if logistic_rows else pd.DataFrame()

logistic_comparison


,key,variable,python_value,sas_value,absolute_difference,relative_difference,pass
0,term=admission_type_DIRECT EMER.,coefficient,1.950838e+01,1.313508e+01,6.373303e+00,3.266956e-01,False
1,term=admission_type_DIRECT OBSERVATION,coefficient,1.672940e+01,1.035610e+01,6.373303e+00,3.809641e-01,False
2,term=admission_type_ELECTIVE,coefficient,1.731894e+01,1.094563e+01,6.373303e+00,3.679962e-01,False
3,term=admission_type_EU OBSERVATION,coefficient,1.681769e+01,1.044438e+01,6.373303e+00,3.789643e-01,False
4,term=admission_type_EW EMER.,coefficient,1.945029e+01,1.307699e+01,6.373303e+00,3.276713e-01,False
...,...,...,...,...,...,...,...
120,term=insurance_Unknown,p_value,1.793694e-09,1.793693e-09,5.870272e-16,3.272728e-07,True
121,term=race_group_Black,p_value,2.272240e-06,2.272238e-06,1.221282e-12,5.374796e-07,True
122,term=race_group_Hispanic,p_value,2.838547e-06,2.838546e-06,1.490193e-12,5.249842e-07,True
123,term=race_group_Other,p_value,1.818855e-02,1.818854e-02,3.426207e-09,1.883717e-07,True


In [79]:
logistic_discrepancies = logistic_comparison.loc[
    logistic_comparison["pass"].eq(False)
].copy() if not logistic_comparison.empty else pd.DataFrame()

logistic_discrepancies


,key,variable,python_value,sas_value,absolute_difference,relative_difference,pass
0,term=admission_type_DIRECT EMER.,coefficient,1.950838e+01,1.313508e+01,6.373303e+00,0.326696,False
1,term=admission_type_DIRECT OBSERVATION,coefficient,1.672940e+01,1.035610e+01,6.373303e+00,0.380964,False
2,term=admission_type_ELECTIVE,coefficient,1.731894e+01,1.094563e+01,6.373303e+00,0.367996,False
3,term=admission_type_EU OBSERVATION,coefficient,1.681769e+01,1.044438e+01,6.373303e+00,0.378964,False
4,term=admission_type_EW EMER.,coefficient,1.945029e+01,1.307699e+01,6.373303e+00,0.327671,False
5,term=admission_type_OBSERVATION ADMIT,coefficient,1.897611e+01,1.260281e+01,6.373303e+00,0.335859,False
6,term=admission_type_SURGICAL SAME DAY ADMISSION,coefficient,1.688063e+01,1.050733e+01,6.373303e+00,0.377551,False
7,term=admission_type_URGENT,coefficient,2.017699e+01,1.380368e+01,6.373303e+00,0.315870,False
13,term=const,coefficient,-2.838314e+01,-2.200984e+01,6.373303e+00,0.224545,False
25,term=admission_type_DIRECT EMER.,odds_ratio,2.967443e+08,5.063981e+05,2.962379e+08,0.998293,False


## Discrepancy Summary

The tables above are intentionally direct: one row per compared field, with Python value, SAS value, absolute difference, relative difference, and a pass/fail flag. Rows that fail tolerance should be reviewed against the upstream Python and SAS source files listed in each section.


In [80]:
summary = pd.DataFrame(
    [
        {
            "stage": "04a unadjusted outcomes",
            "comparison_rows": len(unadjusted_comparison),
            "failed_rows": len(unadjusted_discrepancies),
        },
        {
            "stage": "04b multivariable model",
            "comparison_rows": len(logistic_comparison),
            "failed_rows": len(logistic_discrepancies),
        },
    ]
)
summary


,stage,comparison_rows,failed_rows
0,04a unadjusted outcomes,14,0
1,04b multivariable model,125,35
